# Synthetic Data Pipeline

This notebook runs the synthetic data generation pipeline.

### Before getting started:
- Ensure you have read `docs/*` and `README.md`
- Check that `params.py` and `config.py` are correct.
- Check the `call_LLM` and `red_write_data` functions in `processing.py` are correctly configured for your platform .
- Check that your input data exists and is correctly formatted.

First, import the required classes. 

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
from src.data_generator import generate_patients, generate_admissions, generate_journeys, generate_clinical_notes, add_augmentations, save_final_outputs
from datetime import datetime
from config.params import PARAMS

In [3]:
# INSERT RUN NAME BELOW
# This will be saved in the journey dataset at the end of the notebook for evaluation purposes
run_name = "wp_test_run"
current_time = datetime.now()
version_tag = current_time.strftime("%Y-%m-%d") + f"/{run_name}"
print(f"Run name: {run_name}\nDate: {current_time}\nVersion tag: {version_tag}")

Run name: wp_test_run
Date: 2026-04-24 16:24:53.152052
Version tag: 2026-04-24/wp_test_run


## 1 - Get Patients Information and Admissions

- Generates a list of patients with corresponding admission reasons.


In [4]:
patient_generator = generate_patients()
patients = await patient_generator.run(return_output = True)
patient_generator.write_patients_to_dataset()

Generating 1 patients... ['Lisa', 'Margaret', 'Wallace']
['Douglas', 'Charles', 'Grant']
DONE


In [4]:
admission_generator = generate_admissions()
admissions = await admission_generator.run(return_output = True)
admission_generator.write_admissions_to_dataset()

Generating 1 admissions... ['Ralph', 'Wilfred', 'Greenwood']
DONE


## 2 - Generating and Filtering Journeys

- Generates, validates, and adds details to patient journeys.
- Filtering removes any document types that are not listed as possible event types in `params.py`.

In [4]:
journey_generator = generate_journeys()
journeys = await journey_generator.run(return_outputs = True)
journey_generator.write_journeys_to_dataset()

Generating simple journeys... Validating simple journeys...
Validator Changes:
 {'Patient_0': 0}
Generating extra details... Generating staff personas... Creating full detailed journeys... Filtering journeys...
Patient 0 - Removing events: []
Journey 0 has length 103
DONE


## 3 - Generate and Validate Clinical Notes

- Uses LLMs to generate clinical notes. 
- Validates each note using an LLM Judge. 

In [5]:
clinical_note_generator = generate_clinical_notes()
notes = await clinical_note_generator.run(return_output = True)
clinical_note_generator.write_patient_documents_to_dataset()

Patient 0: Generating Notes... LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error code: 429 on attempt 1/3. Retrying...
LLM call failed with error: Error

/mnt/c/Users/Will Poulett/Documents/synthetic_clinical_notes/src/data_generator.py:405: SyntaxWarning: invalid escape sequence '\s'
  df["ChiefComplaintDescription"] = df["ChiefComplaintDescription"].str.split("\s+\(").str.get(0)
/mnt/c/Users/Will Poulett/Documents/synthetic_clinical_notes/src/data_generator.py:406: SyntaxWarning: invalid escape sequence '\s'
  df["DiagnosisDescription"] = df["DiagnosisDescription"].str.split("\s+\(").str.get(0)


AttributeError: 'str' object has no attribute 'items'

## 4 - Add Augmentations to clinical Notes

This section allows for the augmentatio of clinical notes by:

- Replacing long phrases with abbreviations.
- Adding typos.
- Adding signatures.

In [ ]:
augmentator = add_augmentations()
augmented_notes = await augmentator.run(True)
augmentator.write_final_documents_to_dataset()

## 5 - Write Clinical Notes to Dataset

- Writes the clinical notes to a dataset, alongside the patient journey and admission details.

In [ ]:
output_saver = save_final_outputs()
output_saver.run(run_name, current_time, version_tag)